# Лабораторная работа 7 — Метод Вольфа (Wolfe)

В этом ноутбуке реализован поиск по направлению, удовлетворяющий условиям Вольфа (Wolfe conditions), а также метод градиентного спуска с этим условием выбора шага. Также включён код симплекс-метода из Lab_4 для удобства (если потребуется).

Все пояснения и комментарии даны на русском языке.

In [2]:
import numpy as np
from math import inf

def _phi(f, xk, pk, alpha):
	return f(xk + alpha * pk)

def _phi_grad(grad, xk, pk, alpha):
	return np.dot(grad(xk + alpha * pk), pk)

def wolfe_line_search(f, grad, xk, pk, alpha0=1.0, c1=1e-4, c2=0.9, max_iter=50):
	"""
	Поиск шага по правилу Вольфа (strong Wolfe conditions).

	Вход:
	- f: функция, возвращающая скаляр f(x)
	- grad: функция, возвращающая градиент в точке x (вектор)
	- xk: текущая точка (numpy array)
	- pk: направление спуска (numpy array)
	- alpha0: начальная попытка шага
	- c1, c2: параметры условий Вольфа (обычно c1 ~ 1e-4, c2 ~ 0.9)
	- max_iter: максимальное число итераций

	Возвращает найденный шаг alpha.
	"""
	alpha_prev = 0.0
	alpha = alpha0
	phi0 = _phi(f, xk, pk, 0.0)
	phi_prev = phi0
	phi_grad0 = _phi_grad(grad, xk, pk, 0.0)

	for i in range(max_iter):
		phi_alpha = _phi(f, xk, pk, alpha)
		if (phi_alpha > phi0 + c1 * alpha * phi_grad0) or (i > 0 and phi_alpha >= phi_prev):
			# need to zoom between alpha_prev and alpha
			return _wolfe_zoom(f, grad, xk, pk, alpha_prev, alpha, phi0, phi_grad0, c1, c2)

		phi_grad_alpha = _phi_grad(grad, xk, pk, alpha)
		if abs(phi_grad_alpha) <= -c2 * phi_grad0:
			return alpha

		if phi_grad_alpha >= 0:
			return _wolfe_zoom(f, grad, xk, pk, alpha, alpha_prev, phi0, phi_grad0, c1, c2)

		alpha_prev = alpha
		phi_prev = phi_alpha
		alpha = alpha * 2.0

	return alpha


def _wolfe_zoom(f, grad, xk, pk, alo, ahi, phi0, phi_grad0, c1, c2, max_iter=50):
	"""Вспомогательная процедура zoom для алгоритма Вольфа."""
	for i in range(max_iter):
		# простой бисквичный выбор промежуточного alpha (можно улучшить интерполяцией)
		alpha = 0.5 * (alo + ahi)
		phi_alpha = _phi(f, xk, pk, alpha)
		phi_alo = _phi(f, xk, pk, alo)

		if (phi_alpha > phi0 + c1 * alpha * phi_grad0) or (phi_alpha >= phi_alo):
			ahi = alpha
		else:
			phi_grad_alpha = _phi_grad(grad, xk, pk, alpha)
			if abs(phi_grad_alpha) <= -c2 * phi_grad0:
				return alpha
			if phi_grad_alpha * (ahi - alo) >= 0:
				ahi = alo
			alo = alpha

	return alpha

**Пояснение (на русском):**

Метод Вольфа — это критерий для выбора длины шага при спуске по направлению. Условия Вольфа (strong Wolfe conditions) состоят из двух неравенств: условие Армихо (слабое уменьшение функции) и условие ограничения величины производной по направлению (контроль крутизны). Алгоритм обычно реализуется с процедурой "zoom" для уточнения шага.

In [3]:
def gradient_descent_wolfe(f, grad, x0, tol=1e-6, max_iter=1000, c1=1e-4, c2=0.9):
	xk = np.array(x0, dtype=float)
	history = [xk.copy()]
	for k in range(max_iter):
		gk = grad(xk)
		if np.linalg.norm(gk) < tol:
			return xk, history
		pk = -gk
		alpha = wolfe_line_search(f, grad, xk, pk, alpha0=1.0, c1=c1, c2=c2)
		xk = xk + alpha * pk
		history.append(xk.copy())
	return xk, history

**Короткое напоминание:** код симплекс-метода взят из Lab_4 и добавлен сюда для удобства — он пригодится если в заданиях лабораторной требуется решать задачи линейного программирования.

In [4]:
import numpy as np

def simplex_main_phase(c, A, x_init, B_init, max_iter=1000):
	x = np.array(x_init, dtype=float)
	B = list(B_init)
	for iteration in range(max_iter):
		A_B = A[:, B]
		try:
			A_B_inv = np.linalg.inv(A_B)
		except np.linalg.LinAlgError:
			raise ValueError("Матрица A_B вырождена. B не является базисом.")
		c_B = c[B]
		u = c_B @ A_B_inv
		delta = c - u @ A
		eps = 1e-9
		if np.all(delta <= eps):
			return "optimal", x, B
		j0 = -1
		for j in range(len(delta)):
			if delta[j] > eps:
				j0 = j
				break
		z = A_B_inv @ A[:, j0]
		theta = np.full(len(B), np.inf)
		for i in range(len(B)):
			if z[i] > eps:
				theta[i] = x[B[i]] / z[i]
		theta0 = np.min(theta)
		if np.isinf(theta0):
			return "unbounded", None, B
		k = np.argmin(theta)
		j_star = B[k]
		x[j0] = theta0
		for i in range(len(B)):
			if i != k:
				x[B[i]] = x[B[i]] - theta0 * z[i]
		x[j_star] = 0
		B[k] = j0
	raise RuntimeError("Превышено максимальное число итераций!")

def initial_phase_simplex(c, A, b, max_iter=1000, eps=1e-9):
	A_work = np.array(A, dtype=float).copy()
	b_work = np.array(b, dtype=float).copy()
	c = np.array(c, dtype=float).copy()
	m, n = A_work.shape
	for i in range(m):
		if b_work[i] < 0:
			b_work[i] *= -1
			A_work[i, :] *= -1
	I_m = np.eye(m)
	A_tilde = np.hstack([A_work, I_m])
	c_tilde = np.hstack([np.zeros(n), -np.ones(m)])
	x_tilde0 = np.zeros(n + m)
	x_tilde0[n:] = b_work
	B = list(range(n, n + m))
	status, x_tilde_opt, B = simplex_main_phase(c_tilde, A_tilde, x_tilde0, B, max_iter=max_iter)
	if status != "optimal":
		return {"feasible": False, "message": "Вспомогательная задача не решена до оптимума.", "x": None, "B": None, "A_reduced": None, "b_reduced": None}
	if np.any(x_tilde_opt[n:] > eps):
		return {"feasible": False, "message": "Исходная задача несовместна (допустимых планов нет).", "x": None, "B": None, "A_reduced": None, "b_reduced": None}
	x = x_tilde_opt[:n].copy()
	while any(j >= n for j in B):
		artificial_positions = [idx for idx, val in enumerate(B) if val >= n]
		k = max(artificial_positions, key=lambda idx: B[idx])
		j_k = B[k]
		A_B = A_tilde[:, B]
		A_B_inv = np.linalg.inv(A_B)
		nonbasic_real = [j for j in range(n) if j not in B]
		replacement = None
		for j in nonbasic_real:
			l_j = A_B_inv @ A_tilde[:, j]
			if abs(l_j[k]) > eps:
				replacement = j
				break
		if replacement is not None:
			B[k] = replacement
			continue
		col = A_tilde[:, j_k]
		candidate_rows = np.where(np.abs(col) > eps)[0]
		if len(candidate_rows) == 0:
			row_to_delete = k
		else:
			row_to_delete = int(candidate_rows[0])
		A_work = np.delete(A_work, row_to_delete, axis=0)
		b_work = np.delete(b_work, row_to_delete, axis=0)
		A_tilde = np.delete(A_tilde, row_to_delete, axis=0)
		B.pop(k)
		if len(B) == 0:
			break
	return {"feasible": True, "message": "Исходная задача совместна.", "x": x, "B": B, "A_reduced": A_work, "b_reduced": b_work}

**Примеры и проверка корректности (на стандартных функциях):**

Ниже приведены тесты для квадратичной функции (чья точная точка минимума известна) и пример применения градиентного спуска с поиском шага по Вольфу.

In [5]:
# Тест: квадратичная функция f(x) = 0.5 x^T Q x + b^T x
Q = np.array([[4.0, 1.0],[1.0, 3.0]])
b = np.array([-1.0, -2.0])

def f_quad(x):
	x = np.array(x)
	return 0.5 * x @ Q @ x + b @ x

def grad_quad(x):
	x = np.array(x)
	return Q @ x + b

# Аналитический минимум
Q_inv = np.linalg.inv(Q)
x_star = -Q_inv @ b

print("Аналитическая точка минимума:", np.round(x_star, 6))

# Запустим градиентный спуск с Вольфом
x0 = np.array([0.0, 0.0])
res_x, hist = gradient_descent_wolfe(f_quad, grad_quad, x0, tol=1e-8, max_iter=200)

print("Найденная точка:", np.round(res_x, 6))
print("Расстояние до аналитического минимума:", np.linalg.norm(res_x - x_star))

Аналитическая точка минимума: [0.090909 0.636364]
Найденная точка: [0.090909 0.636364]
Расстояние до аналитического минимума: 6.244632627954484e-09


Если нужно, я могу запустить дополнительные примеры из `lab7.pdf` — пришлите, пожалуйста, конкретный пример или укажите номер примера в PDF, и я оформлю проверку на нём. Также могу скорректировать параметры `c1`, `c2` или добавить визуализацию траектории итераций.